# Assignment 3 — Question 3

For this question I'm building Residue Interaction Graph (RIG) and Long-Range Interaction Network (LIN) models for five proteins from the given list. The idea is to represent each protein as a network where nodes are amino acids and edges represent spatial closeness, then analyse the topological properties of these networks.

## Chosen Proteins

I picked these five proteins because they cover a range of sizes and secondary structure types, which should make the comparison more interesting. All five are well-studied small proteins with known two-state folding kinetics.

| PDB ID | Protein Name | Organism | # Residues | Structure Type |
|--------|-------------|----------|------------|----------------|
| 1UBQ | Ubiquitin | Human | 76 | Mixed α/β |
| 1HRC | Cytochrome C | Horse heart | 104 | Mostly α-helical |
| 1FKB | FKBP12 (FK506-binding protein) | Human | 107 | β-sheet rich |
| 1APS | Acylphosphatase | Human | 98 | Mixed α/β |
| 1CSP | Cold Shock Protein B | *B. subtilis* | 67 | β-barrel |

**Note on data:** PDB files for the five proteins are downloaded automatically from RCSB (https://files.rcsb.org) and saved to the `Question 3/pdb_files/` folder. If a download fails, manually download the `.pdb` file for the relevant protein from https://www.rcsb.org and place it in `Question 3/pdb_files/` using the filename format `<pdbid>.pdb` (e.g. `1ubq.pdb`).

In [ ]:
import os
import urllib.request
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
from itertools import combinations

# Resolve this notebook's directory so all files are saved alongside it
try:
    SAVE_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    SAVE_DIR = os.path.abspath(os.path.dirname(''))
print(f'Save directory: {SAVE_DIR}')

Save directory: /kaggle/working


## 1. Downloading PDB Files

PDB files are downloaded directly from RCSB using the protein IDs. Each file contains the full 3D atomic coordinates for the protein.

In [ ]:
PROTEINS = ['1ubq', '1hrc', '1fkb', '1aps', '1csp']
PDB_DIR  = os.path.join(SAVE_DIR, 'pdb_files')
os.makedirs(PDB_DIR, exist_ok=True)

def download_pdb(pdb_id):
    pdb_id = pdb_id.lower()
    dest = os.path.join(PDB_DIR, f'{pdb_id}.pdb')
    if not os.path.exists(dest):
        url = f'https://files.rcsb.org/download/{pdb_id.upper()}.pdb'
        print(f'Downloading {pdb_id}...')
        urllib.request.urlretrieve(url, dest)
    else:
        print(f'{pdb_id}.pdb already exists.')
    return dest

pdb_paths = {pid: download_pdb(pid) for pid in PROTEINS}
print('All PDB files ready.')

## 2. Extracting Cα Coordinates

I'm parsing the PDB files manually by scanning ATOM records and pulling out the CA (alpha-carbon) atom for each residue. The Cα atom is used as the single representative point for each amino acid — this is the standard approach for building residue contact networks.

In [ ]:
def extract_ca_coords(pdb_file):
    """
    Parse a PDB file and extract Cα coordinates for the first chain of the first MODEL.
    Returns a list of (residue_number, np.array([x, y, z])).
    """
    ca_coords = []
    seen_residues = set()
    first_chain = None
    in_model = False

    with open(pdb_file, 'r') as f:
        for line in f:
            record = line[:6].strip()

            if record == 'MODEL':
                if in_model:          # second MODEL encountered → stop
                    break
                in_model = True
                continue

            if record == 'ENDMDL':
                break

            if record != 'ATOM':
                continue

            atom_name = line[12:16].strip()
            if atom_name != 'CA':
                continue

            chain_id = line[21]
            if first_chain is None:
                first_chain = chain_id
            if chain_id != first_chain:
                continue

            res_seq  = int(line[22:26].strip())
            ins_code = line[26].strip()
            res_key  = (res_seq, ins_code)

            if res_key in seen_residues:
                continue
            seen_residues.add(res_key)

            x = float(line[30:38])
            y = float(line[38:46])
            z = float(line[46:54])
            ca_coords.append((res_seq, np.array([x, y, z])))

    # Sort by residue number
    ca_coords.sort(key=lambda t: t[0])
    return ca_coords


ca_data = {}
for pid in PROTEINS:
    coords = extract_ca_coords(pdb_paths[pid])
    ca_data[pid] = coords
    print(f'{pid}: {len(coords)} residues')

## 3. Building RIG and LIN

**RIG:** Two residues i and j are connected if their Cα atoms are within 8 Å of each other and |i−j| ≥ 2 (excluding direct backbone neighbours, which would be trivially close).

**LIN:** Same 8 Å distance cutoff, but only connects residues that are more than 12 positions apart in the sequence. This filters out local contacts and keeps only long-range spatial interactions.

In [ ]:
RIG_CUTOFF = 8.0   # Angstroms
LIN_SEQ_THRESHOLD = 12

def build_RIG(ca_coords, cutoff=RIG_CUTOFF):
    """Build Residue Interaction Graph (|i-j| >= 2, distance < cutoff)."""
    n = len(ca_coords)
    coords = np.array([c for _, c in ca_coords])
    G = nx.Graph()
    G.add_nodes_from(range(n))
    for i in range(n):
        for j in range(i + 2, n):          # skip direct sequence neighbours
            dist = np.linalg.norm(coords[i] - coords[j])
            if dist < cutoff:
                G.add_edge(i, j, weight=dist)
    return G


def build_LIN(ca_coords, cutoff=RIG_CUTOFF, seq_threshold=LIN_SEQ_THRESHOLD):
    """Build Long-Range Interaction Network (|i-j| > seq_threshold, distance < cutoff)."""
    n = len(ca_coords)
    coords = np.array([c for _, c in ca_coords])
    G = nx.Graph()
    G.add_nodes_from(range(n))
    for i in range(n):
        for j in range(i + seq_threshold + 1, n):   # |i-j| > 12
            dist = np.linalg.norm(coords[i] - coords[j])
            if dist < cutoff:
                G.add_edge(i, j, weight=dist)
    return G


rig_graphs = {}
lin_graphs = {}

for pid in PROTEINS:
    rig_graphs[pid] = build_RIG(ca_data[pid])
    lin_graphs[pid] = build_LIN(ca_data[pid])
    print(f'{pid} | RIG: {rig_graphs[pid].number_of_nodes()} nodes, '
          f'{rig_graphs[pid].number_of_edges()} edges  |  '
          f'LIN: {lin_graphs[pid].number_of_nodes()} nodes, '
          f'{lin_graphs[pid].number_of_edges()} edges')

## 4. Computing Network Properties

In [ ]:
def graph_metrics(G):
    """
    Compute characteristic path length L, average clustering coefficient C,
    and average degree for a graph. L is computed on the largest connected component.
    """
    if G.number_of_edges() == 0:
        return float('nan'), float('nan'), 0.0

    # Average degree
    avg_degree = np.mean([d for _, d in G.degree()])

    # Largest connected component for path length
    lcc_nodes = max(nx.connected_components(G), key=len)
    G_lcc = G.subgraph(lcc_nodes).copy()

    L = nx.average_shortest_path_length(G_lcc)
    C = nx.average_clustering(G)
    return L, C, avg_degree


results = []
for pid in PROTEINS:
    L_rig, C_rig, avg_deg_rig = graph_metrics(rig_graphs[pid])
    L_lin, C_lin, avg_deg_lin = graph_metrics(lin_graphs[pid])
    n_res = len(ca_data[pid])
    results.append({
        'Protein':       pid.upper(),
        'N residues':    n_res,
        'RIG edges':     rig_graphs[pid].number_of_edges(),
        'RIG avg degree':round(avg_deg_rig, 3),
        'RIG L':         round(L_rig, 4),
        'RIG C':         round(C_rig, 4),
        'LIN edges':     lin_graphs[pid].number_of_edges(),
        'LIN avg degree':round(avg_deg_lin, 3),
        'LIN L':         round(L_lin, 4),
        'LIN C':         round(C_lin, 4),
    })
    print(f'{pid.upper()} done.')

df = pd.DataFrame(results)
print(df.to_string(index=False))

## 5. Results

In [ ]:
print('\n=== Characteristic Path Length (L) and Clustering Coefficient (C) ===')
print(df.to_string(index=False))

# Nicely formatted table
try:
    from IPython.display import display
    display(df.style.set_caption('RIG and LIN metrics for five proteins'))
except ImportError:
    pass

## 6. Contact Maps

Visualising the adjacency matrix for each protein's RIG and LIN. Dark squares mean there's an edge between those two residues.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 8))

for col, pid in enumerate(PROTEINS):
    n = len(ca_data[pid])

    for row, (G, label) in enumerate([(rig_graphs[pid], 'RIG'), (lin_graphs[pid], 'LIN')]):
        mat = np.zeros((n, n))
        for (i, j) in G.edges():
            mat[i][j] = 1
            mat[j][i] = 1
        axes[row][col].imshow(mat, cmap='Blues', interpolation='none')
        axes[row][col].set_title(f'{pid.upper()} — {label}', fontsize=9)
        axes[row][col].set_xlabel('Residue', fontsize=7)
        axes[row][col].set_ylabel('Residue', fontsize=7)

plt.suptitle('Contact Maps: RIG (top row) and LIN (bottom row)', fontsize=12)
plt.tight_layout()
out_path = os.path.join(SAVE_DIR, 'contact_maps.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## 7. Observations

**RIG:**
All five proteins show small-world behaviour — short average path lengths and high clustering coefficients. Smaller proteins (1CSP, 1UBQ) have lower L simply because there are fewer residues to travel between. The high C values reflect the fact that residues packed close together in 3D space naturally form dense local neighbourhoods.

**LIN (long-range only):**
LIN is much sparser since we're filtering out all short- and medium-range contacts. For small proteins this can leave the network nearly disconnected. Proteins with β-sheet structure (1FKB, 1CSP) keep more LIN edges because β-sheets bring sequentially distant residues into close spatial proximity.

**Expected folding rate:**
Proteins with more long-range contacts (higher LIN/RIG ratio) tend to fold more slowly — they need a larger conformational search to bring distant residues together. Mostly helical proteins like 1HRC fold faster because their structure is built mostly from local contacts.

| Protein | Dominant structure | Expected folding speed |
|---------|-------------------|----------------------|
| 1UBQ | Mixed α/β | moderate |
| 1HRC | Mostly α | fast |
| 1FKB | β-sheet rich | slow |
| 1APS | Mixed α/β | moderate |
| 1CSP | β-barrel | slow |